# Prueba 4 — Post-filtro + sistema completo vs mono

Última prueba de la narrativa (beam → WPE+beam → **WPE+beam+post**). Sobre el
sistema elegido (NM-MVDR, WPE on) se estudian las variantes de post-filtro y se
compara el sistema completo contra el DTLN monocanal.

- **Sistema completo** = NM-MVDR + BAN + PF (sustracción espectral). El DTLN
  *completo* (2º núcleo) no entra acá; el post-filtro es BAN/PF.
- **Post-filtros:** `PF` y `BAN_PF` con `smooth ∈ {0.2,0.33,0.5,0.66}` + base + BAN.
  DTLN-mono sale automático como baseline single-channel.

**Objetivos:** (a) caracterizar el trade-off y elegir `smooth*`; (b) mostrar el
sistema completo venciendo al DTLN-mono.

Fijos: WPE on (taps=5, delay* de la Prueba 2), estrés espacial FIJO (1 y 3
interferentes), RT60 {160,360,610}, iSIR {0,5,10}, early-only.

**Antes:** poné `WPE_DELAY` = delay* de la Prueba 2.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')


In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive

# CONSTRUIR (git+ para las libs de GitHub).
BUILD = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "git+https://github.com/fgnt/pb_bss.git",
    "git+https://github.com/LCAV/pyroomacoustics.git",
    "git+https://github.com/fgnt/nara_wpe.git",
    "git+https://github.com/fakufaku/fast_bss_eval.git",
]
# INSTALAR desde cache: NOMBRES (no git+, si no pip vuelve a clonar).
INSTALL = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "pb_bss", "pyroomacoustics", "nara_wpe", "fast_bss_eval",
]

# Reconstruye el cache SOLO si la lista de paquetes cambio (manifest) -> se
# autocura si agrego/saco un paquete, sin tener que borrar el cache a mano.
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
need_build = (not os.path.isfile(manifest)) or open(manifest).read() != key

if need_build:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    !pip wheel --wheel-dir=$WHL {" ".join(BUILD)}
    with open(manifest, "w") as fh:
        fh.write(key)
    print("[*] Cache actualizado en", WHL)

!pip install --no-index --find-links=$WHL {" ".join(INSTALL)}
print("[*] Paquetes instalados desde el cache de Drive.")
# Si Colab actualiza Python y falla un import:  !rm -rf $WHL  (se reconstruye solo)

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Config + Ejecución

In [ ]:
import sys, os, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import (NM_MVDR, DTLN_MB_MVDR_SOUDEN_BAN,
                                    NM_MVDR_PF, NM_MVDR_BAN_PF)
from propagation.mird_loader import MirdDatasetProvider

m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite OK (DTLN-mono baseline activo).")
else:
    print("[!] Sin DTLN interpreters -> NO hay baseline DTLN-mono. Cargalos para la comparacion.")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
WPE_DELAY = 2      # <-- REEMPLAZAR con delay* de la Prueba 2
WPE_TAPS  = 5
DURATION  = 15
SMOOTHS   = [0.2, 0.33, 0.5, 0.66]   # trimalo (p.ej. [0.33,0.5]) si tarda mucho
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
]]
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav", "hairdryer_07_SH_MKH800.wav", "drill_07_RHODE_NT1.wav",
]]
# Estres espacial FIJO (no se barre el conteo): 1 interferente y 3 simultaneos.
INTERF_CONFIGS = [
    [(45, 1.0, 0)],
    [(45, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)],
]

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.008,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    'wpe_taps': WPE_TAPS, 'wpe_delay': WPE_DELAY, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1,
    'eval_references': ['early'],
}

param_grid = {
    'rt60':          [0.160, 0.360, 0.610],
    'target_angle':  [0], 'target_dist': [1.0],
    'source_path':   TARGETS,
    'interf_configs':INTERF_CONFIGS,
    'isir_db':       [0, 5, 10],
    'use_wpe':       [True],
    'wpe_taps':      [WPE_TAPS], 'wpe_delay': [WPE_DELAY],
    'mismatch_gain': [0], 'mismatch_phase': [0],
    'error_angle_deg':[0.0], 'error_distance_m':[0.0],
}

# 10 procesadores: base + BAN + PF(smooth) + BAN_PF(smooth). DTLN-mono automatico.
processors_dict = {
    "NM-MVDR": NM_MVDR(min_loading=1e-6, alpha=0.99),
    "BAN":     DTLN_MB_MVDR_SOUDEN_BAN(min_loading=1e-6),
}
for s in SMOOTHS:
    processors_dict[f"PF_{s}"]    = NM_MVDR_PF(min_loading=1e-6, alpha=0.99, smooth=s)
    processors_dict[f"BANPF_{s}"] = NM_MVDR_BAN_PF(min_loading=1e-6, alpha=0.99, smooth=s)

n_cells = 3*len(TARGETS)*len(INTERF_CONFIGS)*3
print("="*60)
print(f"PRUEBA 4 | celdas={n_cells} x {len(processors_dict)} proc | delay*={WPE_DELAY}")
print("procesadores:", list(processors_dict.keys()))
print("="*60)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
temp_dir  = f"/content/results_temp/P4_mono_postfiltro_{RUN_TAG}"
drive_dir = f"/content/drive/MyDrive/Tesis_Beamformers/results/P4_mono_postfiltro_{RUN_TAG}"
os.makedirs(temp_dir, exist_ok=True); os.makedirs(drive_dir, exist_ok=True)

df_P4 = run_mird_grid_search(
    grid_params=param_grid, dataset_provider=provider, processors=processors_dict,
    scene_base_config=base_config, output_dir=temp_dir,
    interpreter_1=interpreter_1, interpreter_2=interpreter_2, save_catalog=False, apply_dtln_post=False,
)
print("\n[INFO] Sincronizando a Drive...")
shutil.copytree(temp_dir, drive_dir, dirs_exist_ok=True)
print(f"[EXITO] Prueba 4 guardada en {drive_dir}")

## Análisis — trade-off de post-filtro y sistema vs mono

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(os.path.join(drive_dir, "mird_benchmark_metrics.csv"))

MET = [("Delta_tot_PESQ_early","PESQ"), ("Delta_tot_STOI_early","STOI"),
       ("Delta_tot_SDR_early","SDR"), ("Delta_tot_SIR_early","SIR"),
       ("Delta_tot_SAR_early","SAR")]
cols = [c for c,_ in MET if c in df.columns]

print("=== Δ end-to-end (media sobre escenas) por procesador -> trade-off ===\n")
tab = df.groupby("processor")[cols].mean().rename(columns=dict(MET)).round(3)
# ordenar por PESQ desc para ver el trade-off
print(tab.sort_values("PESQ", ascending=False).to_string())

# --- Sistema completo vs DTLN-mono (metricas ABSOLUTAS vs early) ---
# mono = dtln_alone_* (columna por celda); sistema = proc_* del ganador.
mono_cols = {f"dtln_alone_{m}_early":m for _,m in
             [("","PESQ"),("","STOI"),("","SDR"),("","SIR"),("","SAR")]}
abs_met = ["PESQ","STOI","SDR","SIR","SAR"]
have_mono = all(f"dtln_alone_{m}_early" in df.columns for m in abs_met)
if have_mono:
    print("\n=== ABSOLUTO (vs early): NM-MVDR base vs DTLN-mono ===")
    row = {}
    for m in abs_met:
        base_sys = df[df.processor=="NM-MVDR"][f"proc_{m}_early"].mean()
        mono     = df[f"dtln_alone_{m}_early"].mean()
        row[m] = (round(base_sys,3), round(mono,3))
    print("  metrica: (NM-MVDR, DTLN-mono)")
    for m,v in row.items(): print(f"   {m:5s}: {v}")
    print("\n(para el sistema COMPLETO usa proc_*_early del BANPF_smooth* en la figura)")
else:
    print("\n[!] Sin columnas dtln_alone_* -> corriste sin DTLN interpreters; no hay mono.")

## Figuras — trade-off (elegir smooth*) y sistema vs mono

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(os.path.join(drive_dir, "mird_benchmark_metrics.csv"))
agg = df.groupby("processor")[[c for c,_ in MET if c in df.columns]].mean()

# --- Fig A: trade-off. x = STOI y SAR ; y = PESQ. Cada punto un procesador. ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, xcol, xlab in [(axes[0],"Delta_tot_STOI_early","Δ STOI"),
                       (axes[1],"Delta_tot_SAR_early","Δ SAR")]:
    if xcol not in agg.columns: continue
    for name, r in agg.iterrows():
        fam = "BANPF" if name.startswith("BANPF") else ("PF" if name.startswith("PF_") else "otro")
        col = {"BANPF":"tab:blue","PF":"tab:red","otro":"gray"}[fam]
        ax.scatter(r[xcol], r["Delta_tot_PESQ_early"], c=col, s=60, zorder=3)
        ax.annotate(name, (r[xcol], r["Delta_tot_PESQ_early"]), fontsize=7,
                    xytext=(4,4), textcoords="offset points")
    ax.set_xlabel(xlab); ax.set_ylabel("Δ PESQ"); ax.grid(alpha=0.3)
fig.suptitle("Prueba 4 — Trade-off del post-filtro (elegir smooth* en la frontera)")
fig.tight_layout(); fig.savefig(os.path.join(drive_dir,"P4_tradeoff.png"), dpi=140, bbox_inches="tight")
plt.show()

# --- Fig B: sistema completo vs mono vs base (ABSOLUTO, barras) ---
SYSTEM_PROC = "BANPF_0.33"   # <-- poner el smooth* elegido de la Fig A
ABS = ["PESQ","STOI","SDR","SIR","SAR"]
if all(f"dtln_alone_{m}_early" in df.columns for m in ABS) and SYSTEM_PROC in df.processor.values:
    base_v = [df[df.processor=="NM-MVDR"][f"proc_{m}_early"].mean() for m in ABS]
    sys_v  = [df[df.processor==SYSTEM_PROC][f"proc_{m}_early"].mean() for m in ABS]
    mono_v = [df[f"dtln_alone_{m}_early"].mean() for m in ABS]
    x = np.arange(len(ABS)); w = 0.26
    fig2, ax = plt.subplots(figsize=(10,5))
    ax.bar(x-w, base_v, w, label="NM-MVDR (base)", color="tab:orange")
    ax.bar(x,   sys_v,  w, label=f"Sistema completo ({SYSTEM_PROC})", color="tab:blue")
    ax.bar(x+w, mono_v, w, label="DTLN-mono", color="tab:gray")
    ax.set_xticks(x); ax.set_xticklabels(ABS); ax.grid(alpha=0.3, axis="y")
    ax.set_ylabel("valor absoluto (vs early)"); ax.legend()
    ax.set_title("Prueba 4 — Sistema completo vs DTLN-mono (media sobre escenas)")
    fig2.tight_layout(); fig2.savefig(os.path.join(drive_dir,"P4_sys_vs_mono.png"), dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("[!] Falta dtln_alone_* o el procesador", SYSTEM_PROC, "- ajusta SYSTEM_PROC / corre con DTLN.")